In [1]:
from dotenv import load_dotenv
import anthropic, json, pandas as pd
from pathlib import Path
import os
import re

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [3]:
load_dotenv()
client = anthropic.Anthropic()  # lee ANTHROPIC_API_KEY del entorno

In [4]:
# df = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding="utf-8")

In [5]:
# df[(df.asignatura == "Mates CCSS") & (df.tema == "funciones ccss" )]

In [6]:
def parse_response(texto):
    texto = re.sub(r"```(?:json)?", "", texto).strip()
    dec = json.JSONDecoder()
    objs, i = [], 0
    while True:
        i = texto.find("{", i)
        if i == -1:
            break
        try:
            obj, fin = dec.raw_decode(texto, i)
            if isinstance(obj, dict):
                objs.append(obj)
            i = fin
        except json.JSONDecodeError:
            i += 1

    if not objs:
        raise ValueError(f"Sin JSON en la respuesta: {texto[:150]!r}")

    tipos, obs = [], []
    for o in objs:
        for t in str(o.get("tipo_ejercicio", "")).split("|"):
            t = t.strip()
            if t and t not in tipos:
                tipos.append(t)
        for x in str(o.get("observaciones", "") or "").split(";"):
            x = x.strip()
            if x and x not in obs:
                obs.append(x)

    orden = {"baja": 0, "media": 1, "alta": 2}
    conf = min((o.get("confianza", "baja") for o in objs),
               key=lambda c: orden.get(c, 0))
    return {"tipo_ejercicio": "|".join(tipos),
            "observaciones": "; ".join(obs),
            "confianza": conf}

In [7]:
def classify(enunciado, system_md, modelo="claude-haiku-4-5-20251001"):
    resp = client.messages.create(
        model=modelo,
        max_tokens=1000,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )
    texto = resp.content[0].text
    try:
        return parse_response(texto)
    except Exception as e:
        # Para depurar: guarda la respuesta cruda junto al error
        raise ValueError(f"{e} | RAW: {texto[:300]!r}") from e

In [8]:
def classify_statements(asignatura, tema, prompt_path, csv_enun = root / "csv/ejercicios_clasificados.csv" ):

    # cargar dataframes
    df_enun = pd.read_csv(csv_enun, encoding = "utf-8")
    # máscara para filtrar asignatura y tema
    mask = (df_enun.asignatura == asignatura) & (df_enun.tema == tema)
    df_to_be_classified = df_enun.loc[mask, ["id", "enunciado"]]

    system_md = Path(prompt_path).read_text(encoding="utf-8")

    # borrar el archivo si existe clasificación previa
    output_file = root / "classifier/response.jsonl"
    
    try:
        if output_file.exists():
            output_file.unlink()
    except Exception as e:
        print("Archivo response.jsonl does not exist.")

    # clasificar las líneas del dataframe para clasificar
    for index, row in df_to_be_classified.iterrows():
    
        try:
            r = classify(row.enunciado, system_md)
            r["id"] = row.id
        except Exception as e:
            r = {"id": row.id, "error": str(e)}
        # important!! use write mode = "a" to add
        # classification of each row to results file
        with output_file.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # leer los ejercicios clasificados desde el json
    df_classified = pd.read_json("classifier/response.jsonl", lines = True)

    # añadir campo de asignatura y tema
    df_classified["asignatura"] = asignatura
    df_classified["tema"] = tema

    return df_classified

In [9]:
df_classified = classify_statements(asignatura="Mates II", tema = "funciones",
                                    prompt_path = root / "classifier/funciones_mates_ii.md" )

In [13]:
df_classified.confianza.value_counts()

confianza
alta     349
media    178
baja       2
Name: count, dtype: int64

In [14]:
df_classified.head()

,tipo_ejercicio,observaciones,confianza,id,asignatura,tema
0,optimización,,alta,mates_ii_funciones_optimización_2026_junio_eje...,Mates II,funciones
1,L'Hopital,L'Hopital calcular a/b,media,mates_ii_funciones_2025_julio_ejercicio_6,Mates II,funciones
2,L'Hopital,L'Hopital calcular a,media,mates_ii_funciones_2025_junio_ejercicio_2,Mates II,funciones
3,calcular coeficientes|asíntotas|monotonía,,alta,mates_ii_funciones_2025_junio_ejercicio_3,Mates II,funciones
4,calcular coeficientes|asíntotas|monotonía,,alta,mates_ii_funciones_asintotas_2025_junio_ejerci...,Mates II,funciones


In [15]:
df_classified.to_csv("csv/ejercicios_clasificados_funciones.csv", index = False, encoding="utf-8")